In [9]:
# PROMPT REPAIR NOTEBOOK
# Goal: for each failure topic, generate prompt / policy improvements.

from pathlib import Path
import os
import json

import pandas as pd
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables from .env
load_dotenv()

# Create OpenAI client using the API key
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY1"))

# Project paths
PROJECT_ROOT = Path("..").resolve()
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

print("Project root:", PROJECT_ROOT)
print("Processed data dir:", DATA_PROCESSED)


Project root: C:\Users\User\Documents\retailmind-prototype
Processed data dir: C:\Users\User\Documents\retailmind-prototype\data\processed


In [10]:
topics_summary_path = DATA_PROCESSED / "uss_english_topics_summary_llm_subset.parquet"
topics_rows_path    = DATA_PROCESSED / "uss_english_topics_rows_llm_subset.parquet"
turns_topics_path   = DATA_PROCESSED / "uss_english_turns_topics_llm_subset.parquet"  # optional

df_topics_summary = pd.read_parquet(topics_summary_path)
df_topics_rows = pd.read_parquet(topics_rows_path)

print("df_topics_summary:", df_topics_summary.shape)
print("df_topics_rows:", df_topics_rows.shape)

display(df_topics_summary.head())


df_topics_summary: (4, 6)
df_topics_rows: (506, 9)


,topic_id,topic_label,n_examples,top_issues,example_texts,example_reason
0,0,Topic 0: MISSING_CONTEXT / UNSUPPORTED_INTENT,153,"[MISSING_CONTEXT, UNSUPPORTED_INTENT, WRONG_FA...","[I don't look for to own them anytime soon., I...",The bot fails to acknowledge the user's disint...
1,1,Topic 1: MISSING_CONTEXT / UNSUPPORTED_INTENT,144,"[MISSING_CONTEXT, UNSUPPORTED_INTENT, WRONG_FA...","[No, I'm sorry. I'll be checking in on the 3rd...",The bot incorrectly confirmed the check-in dat...
2,2,Topic 2: MISSING_CONTEXT / TONE_ISSUE,123,"[MISSING_CONTEXT, TONE_ISSUE, UNSUPPORTED_INTENT]","[Oh, I dislike it. It's I don't know if it's o...",The bot fails to acknowledge the user's strong...
3,3,Topic 3: MISSING_CONTEXT / UNSUPPORTED_INTENT,86,"[MISSING_CONTEXT, UNSUPPORTED_INTENT, TONE_ISS...","[Nope, haven't seen that., Don't really know w...",The bot fails to acknowledge the user's previo...


In [11]:
required_summary_cols = {"topic_id", "topic_label", "n_examples", "top_issues", "example_texts", "example_reason"}
missing = required_summary_cols - set(df_topics_summary.columns)
print("Missing summary cols:", missing)

required_rows_cols = {"dataset", "conv_id", "turn_id", "topic_id", "topic_label"}
missing_rows = required_rows_cols - set(df_topics_rows.columns)
print("Missing rows cols:", missing_rows)


Missing summary cols: set()
Missing rows cols: set()


In [12]:
import ast

def normalize_list_field(val):
    """
    Returns a list[str] for any of:
    - numpy array
    - python list
    - stringified list like "['A','B']"
    - single string
    - NaN/None
    """
    if isinstance(val, np.ndarray):
        return [str(x) for x in val.tolist()]

    if isinstance(val, list):
        return [str(x) for x in val]

    if isinstance(val, str):
        s = val.strip()
        # Try parsing list-like strings
        if s.startswith("[") and s.endswith("]"):
            try:
                parsed = ast.literal_eval(s)
                if isinstance(parsed, list):
                    return [str(x) for x in parsed]
            except Exception:
                pass
        return [s] if s else []

    if val is None or (isinstance(val, float) and np.isnan(val)):
        return []

    # fallback
    return [str(val)]


In [13]:
required_summary_cols = {"topic_id", "topic_label", "n_examples", "top_issues", "example_texts", "example_reason"}
missing = required_summary_cols - set(df_topics_summary.columns)
if missing:
    raise ValueError(f"df_topics_summary missing columns: {missing}")

# Normalize list-like columns
df_topics_summary["top_issues"] = df_topics_summary["top_issues"].apply(normalize_list_field)
df_topics_summary["example_texts"] = df_topics_summary["example_texts"].apply(normalize_list_field)

# Ensure example_reason is string (avoid NaN)
df_topics_summary["example_reason"] = df_topics_summary["example_reason"].fillna("").astype(str)

print("Normalization done.")
display(df_topics_summary.head(3))


Normalization done.


,topic_id,topic_label,n_examples,top_issues,example_texts,example_reason
0,0,Topic 0: MISSING_CONTEXT / UNSUPPORTED_INTENT,153,"[MISSING_CONTEXT, UNSUPPORTED_INTENT, WRONG_FA...","[I don't look for to own them anytime soon., I...",The bot fails to acknowledge the user's disint...
1,1,Topic 1: MISSING_CONTEXT / UNSUPPORTED_INTENT,144,"[MISSING_CONTEXT, UNSUPPORTED_INTENT, WRONG_FA...","[No, I'm sorry. I'll be checking in on the 3rd...",The bot incorrectly confirmed the check-in dat...
2,2,Topic 2: MISSING_CONTEXT / TONE_ISSUE,123,"[MISSING_CONTEXT, TONE_ISSUE, UNSUPPORTED_INTENT]","[Oh, I dislike it. It's I don't know if it's o...",The bot fails to acknowledge the user's strong...


In [14]:
import numpy as np
import ast


def build_topic_description(summary_row, extra_examples=None, max_examples: int = 5) -> str:
    """
    Build a textual description of a topic to send to the LLM.
    summary_row: one row from df_topics_summary (already normalized upstream)
    """
    topic_id = int(summary_row["topic_id"])
    topic_label = str(summary_row.get("topic_label", "") or "").strip()
    n_examples = int(summary_row.get("n_examples", 0))

    # Top issues (already normalized, but keep defensive)
    top_issues = normalize_list_field(summary_row.get("top_issues", []))
    top_issues_str = ", ".join(top_issues) if len(top_issues) > 0 else "[none]"

    # Example texts (already normalized, but keep defensive)
    example_texts = normalize_list_field(summary_row.get("example_texts", []))

    # Add extra examples if provided
    if extra_examples:
        example_texts.extend([str(x) for x in extra_examples])

    # Limit examples
    example_texts = example_texts[:max_examples]

    # Example reason: ensure safe string
    example_reason = summary_row.get("example_reason", "")
    if example_reason is None:
        example_reason = ""
    # If it somehow comes as ndarray, join it
    if isinstance(example_reason, np.ndarray):
        example_reason = " ".join([str(x) for x in example_reason.tolist()])
    example_reason = str(example_reason).strip()

    # Build description
    lines = []
    lines.append(f"Topic ID: {topic_id}")
    if topic_label:
        lines.append(f"Topic label: {topic_label}")
    lines.append(f"Number of examples: {n_examples}")
    lines.append(f"Most frequent issue labels: {top_issues_str}")
    lines.append("")
    lines.append("Representative user messages:")

    if len(example_texts) > 0:
        for i, txt in enumerate(example_texts, start=1):
            lines.append(f"{i}) {txt}")
    else:
        lines.append("[No example texts available]")

    if example_reason:
        lines.append("")
        lines.append("Representative diagnostic reason:")
        lines.append(example_reason)

    return "\n".join(lines)


# Quick check
desc_example = build_topic_description(df_topics_summary.iloc[0])
print(desc_example)


Topic ID: 0
Topic label: Topic 0: MISSING_CONTEXT / UNSUPPORTED_INTENT
Number of examples: 153
Most frequent issue labels: MISSING_CONTEXT, UNSUPPORTED_INTENT, WRONG_FACT, HANDOFF_REQUIRED

Representative user messages:
1) I don't look for to own them anytime soon.
2) I'd like to find a movie nearby.
3) I am not well and I need to find a doctor in San Jose.
4) I want to watch a movie around the area.
5) Fine me a restaurant to eat at.

Representative diagnostic reason:
The bot fails to acknowledge the user's disinterest in the topic, leading to a disconnect in the conversation. It should have picked up on the user's lack of enthusiasm and adjusted its response accordingly.


In [15]:
REPAIR_KEYS = [
    "root_cause",
    "suggested_prompt_changes",
    "system_prompt_snippet",
    "guardrail_rules",
    "evaluation_checks",
]

def build_prompt_repair_instruction(topic_description: str) -> str:
    """
    Build the prompt we send to the LLM to generate a repair package for a topic.

    Note:
    - We do NOT ask the model to generate topic_label; topic_id/topic_label are already decided upstream
      (topic_clustering) and must remain stable for the dashboard.
    """
    return f"""
You are a senior conversation designer / LLM product engineer for virtual assistants.

You are given a TOPIC representing a cluster of low-satisfaction user turns.
The topic description includes:
- frequent issue labels (from this taxonomy: MISSING_CONTEXT, WRONG_FACT, TONE_ISSUE, LOOP, UNSUPPORTED_INTENT, SLOW_RESPONSE, HANDOFF_REQUIRED, SUCCESS_BEST_PRACTICE),
- representative user messages,
- an example diagnostic reason extracted from logs.

Your task:
1) Write the most likely root cause behind this failure pattern.
2) Suggest 3–6 concrete prompt/policy changes that a developer can implement.
3) Write a short system_prompt_snippet (6–12 lines) that could be added to the assistant's system prompt.
4) Provide 3–6 guardrail_rules (simple constraints / if-then checks).
5) Provide 3–6 evaluation_checks (small tests to validate the fix).

STRICT OUTPUT REQUIREMENTS:
- Output MUST be valid JSON only (no markdown, no extra text).
- Use exactly these keys: {", ".join(REPAIR_KEYS)}
- suggested_prompt_changes, guardrail_rules, evaluation_checks must be JSON arrays of strings.

TOPIC description:
---
{topic_description}
---

Return JSON only.
""".strip()



In [16]:
def generate_prompt_repair(
    topic_description: str,
    model_name: str = "gpt-4o-mini",
    temperature: float = 0.2,
    max_retries: int = 3,
    sleep_seconds: float = 2.0,
) -> dict:
    """
    Call OpenAI to generate a repair package for a topic.
    Returns a dict with exactly: root_cause, suggested_prompt_changes,
    system_prompt_snippet, guardrail_rules, evaluation_checks.
    """
    prompt = build_prompt_repair_instruction(topic_description)

    for attempt in range(1, max_retries + 1):
        try:
            response = client.chat.completions.create(
                model=model_name,
                temperature=temperature,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": "Return only valid JSON."},
                    {"role": "user", "content": prompt},
                ],
            )

            content = response.choices[0].message.content
            data = json.loads(content)

            # Keep only expected keys (protect downstream schema)
            cleaned = {k: data.get(k, None) for k in REPAIR_KEYS}

            # Defaults for missing
            cleaned["root_cause"] = str(cleaned.get("root_cause") or "").strip()
            cleaned["system_prompt_snippet"] = str(cleaned.get("system_prompt_snippet") or "").strip()

            for k in ["suggested_prompt_changes", "guardrail_rules", "evaluation_checks"]:
                cleaned[k] = normalize_list_field(cleaned.get(k, []))
                cleaned[k] = [str(x).strip() for x in cleaned[k] if str(x).strip()]

            return cleaned

        except Exception as e:
            print(f"[generate_prompt_repair] attempt {attempt}/{max_retries} failed: {e}")
            time.sleep(sleep_seconds)

    # Fallback if all retries fail (keeps pipeline running)
    return {
        "root_cause": "Repair generation failed due to API/parse error.",
        "suggested_prompt_changes": [],
        "system_prompt_snippet": "",
        "guardrail_rules": [],
        "evaluation_checks": [],
    }


In [17]:
repairs = []

for _, row in df_topics_summary.iterrows():
    topic_id = int(row["topic_id"])
    topic_label = str(row.get("topic_label", "") or "").strip()

    # Optional: gather a few extra example texts for this topic (if available)
    extra_examples = None
    if "text" in df_topics_rows.columns:
        extra_examples = (
            df_topics_rows.loc[df_topics_rows["topic_id"] == topic_id, "text"]
            .astype(str)
            .head(3)
            .tolist()
        )

    topic_desc = build_topic_description(row, extra_examples=extra_examples)
    result = generate_prompt_repair(topic_desc)

    repairs.append({
        "topic_id": topic_id,
        "topic_label": topic_label,
        **result
    })

df_repairs = pd.DataFrame(repairs)
display(df_repairs.head())
print("df_repairs shape:", df_repairs.shape)



,topic_id,topic_label,root_cause,suggested_prompt_changes,system_prompt_snippet,guardrail_rules,evaluation_checks
0,0,Topic 0: MISSING_CONTEXT / UNSUPPORTED_INTENT,The assistant is not effectively recognizing u...,[Implement context awareness to better underst...,You should recognize when users express disint...,"[If the user expresses disinterest, provide a ...",[Test the assistant's ability to recognize dis...
1,1,Topic 1: MISSING_CONTEXT / UNSUPPORTED_INTENT,The assistant lacks the ability to maintain co...,[Implement context tracking to remember user i...,You must maintain context throughout the conve...,"[If a user mentions a date, store it in contex...",[Test if the assistant can recall previously m...
2,2,Topic 2: MISSING_CONTEXT / TONE_ISSUE,The assistant fails to recognize and engage wi...,[Implement a tone detection mechanism to ident...,You are an empathetic virtual assistant. Alway...,"[If the user expresses a negative sentiment, r...",[Test if the assistant acknowledges user emoti...
3,3,Topic 3: MISSING_CONTEXT / UNSUPPORTED_INTENT,The assistant is not effectively retaining or ...,[Implement context retention mechanisms to rem...,You are a virtual assistant that prioritizes u...,"[If a user expresses disinterest in a topic, d...",[Test if the assistant retains context across ...


df_repairs shape: (4, 7)


In [18]:
# --- Step: Validate df_repairs schema and basic quality ---

expected_cols = [
    "topic_id",
    "topic_label",
    "root_cause",
    "suggested_prompt_changes",
    "system_prompt_snippet",
    "guardrail_rules",
    "evaluation_checks",
]

missing = [c for c in expected_cols if c not in df_repairs.columns]
extra = [c for c in df_repairs.columns if c not in expected_cols]

print("Missing cols:", missing)
print("Extra cols:", extra)

assert len(missing) == 0, f"df_repairs is missing columns: {missing}"

# Ensure types are as expected
list_cols = ["suggested_prompt_changes", "guardrail_rules", "evaluation_checks"]
for c in list_cols:
    df_repairs[c] = df_repairs[c].apply(normalize_list_field)

# Ensure text fields are strings (no NaN)
for c in ["topic_label", "root_cause", "system_prompt_snippet"]:
    df_repairs[c] = df_repairs[c].fillna("").astype(str)

# Quick checks
print("\nNull counts:")
display(df_repairs[expected_cols].isna().sum())

print("\nList field type counts (first 3 list cols):")
for c in list_cols:
    type_counts = df_repairs[c].apply(type).value_counts()
    print(f"{c}:")
    display(type_counts)

print("\nExample snippet lengths:")
display(df_repairs["system_prompt_snippet"].str.len().describe())


Missing cols: []
Extra cols: []

Null counts:


topic_id                    0
topic_label                 0
root_cause                  0
suggested_prompt_changes    0
system_prompt_snippet       0
guardrail_rules             0
evaluation_checks           0
dtype: int64


List field type counts (first 3 list cols):
suggested_prompt_changes:


suggested_prompt_changes
<class 'list'>    4
Name: count, dtype: int64

guardrail_rules:


guardrail_rules
<class 'list'>    4
Name: count, dtype: int64

evaluation_checks:


evaluation_checks
<class 'list'>    4
Name: count, dtype: int64


Example snippet lengths:


count      4.000000
mean     228.750000
std       10.688779
min      214.000000
25%      224.500000
50%      231.500000
75%      235.750000
max      238.000000
Name: system_prompt_snippet, dtype: float64

In [19]:
# --- Step: Save repair package outputs ---

repairs_parquet_path = DATA_PROCESSED / "uss_english_prompt_repairs_llm_subset.parquet"
repairs_json_path = DATA_PROCESSED / "uss_english_prompt_repairs_llm_subset.json"

df_repairs.to_parquet(repairs_parquet_path, index=False)

# JSON-friendly export (lists stay lists)
df_repairs.to_json(repairs_json_path, orient="records", force_ascii=False, indent=2)

print("Saved:")
print(" -", repairs_parquet_path)
print(" -", repairs_json_path)


Saved:
 - C:\Users\User\Documents\retailmind-prototype\data\processed\uss_english_prompt_repairs_llm_subset.parquet
 - C:\Users\User\Documents\retailmind-prototype\data\processed\uss_english_prompt_repairs_llm_subset.json


In [20]:
# --- Step: Sanity preview of repair packages ---

cols_preview = [
    "topic_id", "topic_label",
    "root_cause",
    "suggested_prompt_changes",
    "system_prompt_snippet",
    "guardrail_rules",
    "evaluation_checks",
]

display(df_repairs[cols_preview])

print("\nExample: Topic 0 repair package")
row0 = df_repairs.sort_values("topic_id").iloc[0].to_dict()
for k in cols_preview:
    print(f"\n{k}:\n{row0.get(k)}")


,topic_id,topic_label,root_cause,suggested_prompt_changes,system_prompt_snippet,guardrail_rules,evaluation_checks
0,0,Topic 0: MISSING_CONTEXT / UNSUPPORTED_INTENT,The assistant is not effectively recognizing u...,[Implement context awareness to better underst...,You should recognize when users express disint...,"[If the user expresses disinterest, provide a ...",[Test the assistant's ability to recognize dis...
1,1,Topic 1: MISSING_CONTEXT / UNSUPPORTED_INTENT,The assistant lacks the ability to maintain co...,[Implement context tracking to remember user i...,You must maintain context throughout the conve...,"[If a user mentions a date, store it in contex...",[Test if the assistant can recall previously m...
2,2,Topic 2: MISSING_CONTEXT / TONE_ISSUE,The assistant fails to recognize and engage wi...,[Implement a tone detection mechanism to ident...,You are an empathetic virtual assistant. Alway...,"[If the user expresses a negative sentiment, r...",[Test if the assistant acknowledges user emoti...
3,3,Topic 3: MISSING_CONTEXT / UNSUPPORTED_INTENT,The assistant is not effectively retaining or ...,[Implement context retention mechanisms to rem...,You are a virtual assistant that prioritizes u...,"[If a user expresses disinterest in a topic, d...",[Test if the assistant retains context across ...



Example: Topic 0 repair package

topic_id:
0

topic_label:
Topic 0: MISSING_CONTEXT / UNSUPPORTED_INTENT

root_cause:
The assistant is not effectively recognizing user intent and context, leading to irrelevant responses and a lack of engagement.

suggested_prompt_changes:
['Implement context awareness to better understand user disinterest and adjust responses accordingly.', 'Enhance intent recognition algorithms to accurately identify unsupported intents and provide appropriate feedback.', 'Incorporate user sentiment analysis to gauge enthusiasm and tailor responses to match user mood.', 'Add fallback responses for unsupported intents that guide users toward relevant topics.', 'Create a more dynamic dialogue flow that allows for context switching based on user cues.']

system_prompt_snippet:
You should recognize when users express disinterest or provide unclear requests. Adjust your responses to be more engaging and contextually relevant. Always seek to clarify user intent and offer a

In [21]:
# --- Optional: dashboard-friendly strings (if needed) ---

df_repairs_dashboard = df_repairs.copy()

for c in ["suggested_prompt_changes", "guardrail_rules", "evaluation_checks"]:
    df_repairs_dashboard[c] = df_repairs_dashboard[c].apply(
        lambda xs: " | ".join(xs) if isinstance(xs, list) else ""
    )

repairs_dashboard_json_path = DATA_PROCESSED / "uss_english_prompt_repairs_llm_subset_dashboard.json"
df_repairs_dashboard.to_json(repairs_dashboard_json_path, orient="records", force_ascii=False, indent=2)

print("Saved dashboard-friendly JSON:")
print(" -", repairs_dashboard_json_path)


Saved dashboard-friendly JSON:
 - C:\Users\User\Documents\retailmind-prototype\data\processed\uss_english_prompt_repairs_llm_subset_dashboard.json
